# UniCrypto — Python quickstart

`unicrypto` is a Cython extension over the UniCrypto C ABI, shipped as a
self-contained wheel: the native library travels inside the package, so
installing it needs neither Nim nor a compiler.

```
pip install unicrypto
```

CI executes this notebook against the wheel the release actually publishes, so
an output below that stops matching fails the build.

## The API

In [1]:
import unicrypto

unicrypto.version(), unicrypto.__version__

('0.1.0', '0.1.0')

`caesar_encrypt` shifts each Latin letter by *shift* positions,
preserving case and passing every other byte through unchanged. `caesar_decrypt`
is its inverse at the same key.

In [2]:
unicrypto.caesar_encrypt("Hello, World!", 13)
unicrypto.caesar_decrypt("Uryyb, Jbeyq!", 13)

'Hello, World!'

## Any integer shift is accepted

The shift is taken modulo 26, so ROT-13 is its own inverse: applying it twice
returns the original text.

In [3]:
s = "Hello, World!"
assert unicrypto.caesar_encrypt(unicrypto.caesar_encrypt(s, 13), 13) == s

In [4]:
# Shifts larger than 26 wrap around.
assert unicrypto.caesar_encrypt("Hello", 3) == unicrypto.caesar_encrypt("Hello", 3 + 26)

Decrypt reverses encrypt at the same key, for any shift — the
contract the Nim library states as a postcondition, expressed here as a
round-trip.

In [5]:
plain = "The quick brown fox jumps over the lazy dog 42!"
assert unicrypto.caesar_decrypt(unicrypto.caesar_encrypt(plain, 7), 7) == plain

A non-string text or non-integer shift is a type error, not a coercion.

In [6]:
try:
    unicrypto.caesar_encrypt(123, 5)
except TypeError as exc:
    print("TypeError:", exc)
else:
    raise AssertionError("expected TypeError")

TypeError: text must be str, got int


In [7]:
try:
    unicrypto.caesar_encrypt("hello", 3.0)
except TypeError as exc:
    print("TypeError:", exc)
else:
    raise AssertionError("expected TypeError")

TypeError: shift must be int, got float


## The C ABI underneath

The same entry points are reachable from anything that speaks C. There the
contract is expressed by returning an error code instead of raising — an
exception must never unwind across an ABI boundary:

```c
ucr_caesar_encrypt("Hello", 13, buf, sizeof buf);   /* bytes written, or -1 */
```

See `include/UniCrypto.h`, and the book for the full picture.

## BLAKE3

`unicrypto`'s second module is BLAKE3, a cryptographic hash function.
Unlike Caesar, its input is `bytes`, not `str` — a hash covers arbitrary
data, which may contain any byte value at all.

In [8]:
unicrypto.blake3_hash(b"abc").hex()

'6437b3ac38465133ffb63b75273a8db548c558465d79db03fd359c6cd5bd9d85'

That 64-character hex string is one of BLAKE3's own official
test vectors: every conforming implementation produces this exact digest
for the three bytes `b"abc"`.

### Extended output (XOF)

The default output is 32 bytes, but any length can be requested. The first
32 bytes of a longer output always equal the default-length hash of the
same input — never a different value.

In [9]:
assert unicrypto.blake3_hash_xof(b"abc", 64)[:32] == unicrypto.blake3_hash(b"abc")

### Keyed hash (MAC)

A 32-byte key turns BLAKE3 into a message authentication code: proof the
caller knew the key, not just the data.

In [10]:
key = bytes(range(32))
unicrypto.blake3_keyed_hash(b"message", key).hex()

'0978071f9c601ec34611813742454dd142c63ffb2cacac65a20b38253323bb00'

Flipping a single bit of the key changes the entire output —
there is no partial credit for an almost-right key.

In [11]:
wrong_key = bytes([key[0] ^ 1]) + key[1:]
assert unicrypto.blake3_keyed_hash(b"message", key) != unicrypto.blake3_keyed_hash(b"message", wrong_key)

### Key derivation

`blake3_derive_key` turns key material into a new key for a hardcoded,
application-specific context string — deriving many purpose-specific keys
from one master secret.

In [12]:
material = b"some master secret"
unicrypto.blake3_derive_key("my app 2026 session keys", material).hex()

'17b8e671d78ae8af680c31b9702fc3f28f0c8dea929b120a81f152ee6a037d5f'

A non-bytes message or a wrong-length key is a type or value
error, not silent truncation or padding.

In [13]:
try:
    unicrypto.blake3_hash("not bytes")
except TypeError as exc:
    print("TypeError:", exc)
else:
    raise AssertionError("expected TypeError")

TypeError: data must be bytes, got str


In [14]:
try:
    unicrypto.blake3_keyed_hash(b"message", bytes(31))  # one byte short
except ValueError as exc:
    print("ValueError:", exc)
else:
    raise AssertionError("expected ValueError")

ValueError: key must be exactly 32 bytes


## The C ABI and CLI underneath

```c
int ucr_blake3_hash(const uint8_t *input, size_t input_len,
                    uint8_t output[UCR_BLAKE3_OUT_LEN]);
```

The input is an explicit-length byte buffer (a pointer and a separate
length), never a NUL-terminated `cstring` like the Caesar procs — a hash's
input may legally contain a zero byte. The CLI favours files and piped
input over inline text, since nobody types raw bytes at a shell prompt:

```bash
unicrypto_cli blake3 -i document.pdf
```

See `include/UniCrypto.h` and the book for the full picture.